# AES-128 — ECB, CBC, PCBC, CFB, OFB e CTR
Execute as células em ordem. Cada exemplo utiliza os mesmos dados e valida a descriptografia.

In [2]:
%pip install pycryptodome

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 21.9 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip available: 22.3.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from Crypto.Cipher import AES

key = bytes.fromhex("000102030405060708090a0b0c0d0e0f")
plaintext = bytes.fromhex("00112233445566778899aabbccddeeff")
iv = bytes.fromhex("101112131415161718191a1b1c1d1e1f")

# Exemplo de 2 blocos para testar encadeamento:
plaintext_2blocks = plaintext + bytes.fromhex("ffeeddccbbaa99887766554433221100")

def show_result(mode, original, encrypted, decrypted):
    print(f"========= AES-128 {mode} =========")
    print("Key:       ", key.hex())
    print("IV:        ", iv.hex() if mode != 'ECB' else 'Não utilizado')
    print("Plaintext: ", original.hex())
    print("Ciphertext:", encrypted.hex())
    print("Decrypted: ", decrypted.hex())
    print("Match:     ", decrypted == original)
    assert decrypted == original, f"Falha no modo {mode}"
    print()

def xor_bytes(a, b):
    if len(a) != len(b):
        raise ValueError('Os operandos XOR precisam ter o mesmo tamanho')
    return bytes(x ^ y for x, y in zip(a, b))

## 1. ECB — Criptografia e descriptografia

In [5]:
ciphertext_ecb = AES.new(key, AES.MODE_ECB).encrypt(plaintext)
decrypted_ecb = AES.new(key, AES.MODE_ECB).decrypt(ciphertext_ecb)
show_result("ECB", plaintext, ciphertext_ecb, decrypted_ecb)
assert ciphertext_ecb.hex() == "69c4e0d86a7b0430d8cdb78070b4c55a"

========= AES-128 ECB =========
Key:        000102030405060708090a0b0c0d0e0f
IV:         Não utilizado
Plaintext:  00112233445566778899aabbccddeeff
Ciphertext: 69c4e0d86a7b0430d8cdb78070b4c55a
Decrypted:  00112233445566778899aabbccddeeff
Match:      True



## 2. CBC — Criptografia e descriptografia

In [6]:
ciphertext_cbc = AES.new(key, AES.MODE_CBC, iv=iv).encrypt(plaintext_2blocks)
decrypted_cbc = AES.new(key, AES.MODE_CBC, iv=iv).decrypt(ciphertext_cbc)
show_result("CBC", plaintext_2blocks, ciphertext_cbc, decrypted_cbc)

========= AES-128 CBC =========
Key:        000102030405060708090a0b0c0d0e0f
IV:         101112131415161718191a1b1c1d1e1f
Plaintext:  00112233445566778899aabbccddeeffffeeddccbbaa99887766554433221100
Ciphertext: 1eca870ffea114b7fd6cf363c30b96b1365d59611eeb7fd4b4c56ddd5a462cb5
Decrypted:  00112233445566778899aabbccddeeffffeeddccbbaa99887766554433221100
Match:      True



## 3. PCBC — Implementação manual do encadeamento

In [ ]:
def pcbc_encrypt(data, key, iv):
    if len(data) % 16:
        raise ValueError('PCBC exige múltiplos de 16 bytes neste exemplo')
    aes = AES.new(key, AES.MODE_ECB)
    feedback = iv
    output = bytearray()
    for offset in range(0, len(data), 16):
        p = data[offset:offset+16]
        c = aes.encrypt(xor_bytes(p, feedback))
        output.extend(c)
        feedback = xor_bytes(p, c)
    return bytes(output)

def pcbc_decrypt(data, key, iv):
    if len(data) % 16:
        raise ValueError('PCBC exige múltiplos de 16 bytes neste exemplo')
    aes = AES.new(key, AES.MODE_ECB)
    feedback = iv
    output = bytearray()
    for offset in range(0, len(data), 16):
        c = data[offset:offset+16]
        p = xor_bytes(aes.decrypt(c), feedback)
        output.extend(p)
        feedback = xor_bytes(p, c)
    return bytes(output)

ciphertext_pcbc = pcbc_encrypt(plaintext_2blocks, key, iv)
decrypted_pcbc = pcbc_decrypt(ciphertext_pcbc, key, iv)
show_result("PCBC", plaintext_2blocks, ciphertext_pcbc, decrypted_pcbc)

## 4. CFB — Segmento de 128 bits

In [ ]:
ciphertext_cfb = AES.new(key, AES.MODE_CFB, iv=iv, segment_size=128).encrypt(plaintext_2blocks)
decrypted_cfb = AES.new(key, AES.MODE_CFB, iv=iv, segment_size=128).decrypt(ciphertext_cfb)
show_result("CFB", plaintext_2blocks, ciphertext_cfb, decrypted_cfb)

## 5. OFB — Criptografia e descriptografia

In [ ]:
ciphertext_ofb = AES.new(key, AES.MODE_OFB, iv=iv).encrypt(plaintext_2blocks)
decrypted_ofb = AES.new(key, AES.MODE_OFB, iv=iv).decrypt(ciphertext_ofb)
show_result("OFB", plaintext_2blocks, ciphertext_ofb, decrypted_ofb)

## 6. CTR — Convenção do MiniSoC (IV XOR contador de 64 bits zero-estendido)

In [ ]:
def minisoc_ctr_crypt(data, key, iv, counter_start=0):
    aes = AES.new(key, AES.MODE_ECB)
    output = bytearray()
    for block_index, offset in enumerate(range(0, len(data), 16)):
        block = data[offset:offset+16]
        counter = (counter_start + block_index) & ((1 << 64) - 1)
        counter_128 = counter.to_bytes(16, byteorder='big')
        aes_input = xor_bytes(iv, counter_128)
        keystream = aes.encrypt(aes_input)
        output.extend(xor_bytes(block, keystream[:len(block)]))
    return bytes(output)

ciphertext_ctr = minisoc_ctr_crypt(plaintext_2blocks, key, iv)
decrypted_ctr = minisoc_ctr_crypt(ciphertext_ctr, key, iv)
show_result("CTR", plaintext_2blocks, ciphertext_ctr, decrypted_ctr)

## Verificação final

In [ ]:
print("Todos os 6 modos executados e validados com sucesso!")